# TerraVista – Member 3: PlantVillage test evaluation + near-duplicate check

Runs **once** and produces **one zip** (`/kaggle/working/terravista_member3_test_evaluation.zip`) for the next person.

**Before you run (Kaggle setup, 2 minutes)**
1. Make one private Kaggle dataset and upload into it: `split__1_.csv` (the one with 7,978 test rows), `resnet50_p10_finetune_checkpoints.zip`, `MobileNetV3_checkpoints.zip` and, optional but recommended, `terravista_finetune_p10_logs_predictions.zip` + `MobileNetV3.zip` (they let the notebook check its own loading against the saved validation predictions).
2. In this notebook press **Add Input** and add: that dataset **and** the *PlantVillage Dataset* by Abdallah Ali.
3. Settings: Accelerator **GPU T4**, Internet **On**.
4. **Run All.** Do not edit anything in the cells.

What it does, in order: find the files → check the models on the **validation** set (stops if they don't reproduce the team's numbers) → freeze the near-duplicate rule → run the test set once per model per seed → statistics → zip.

In [1]:
import os, sys, json, time, hashlib, shutil, zipfile, platform, warnings
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F, torchvision
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from sklearn.metrics import f1_score
warnings.filterwarnings("ignore")

WORK = "/kaggle/working"
OUT = WORK + "/terravista_member3_test_evaluation"
for sub in ("", "/predictions", "/near_duplicate"):
    os.makedirs(OUT + sub, exist_ok=True)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type == "cuda", "No GPU. Settings > Accelerator > GPU T4, then run again."

LOG = []
def log(*a):
    s = " ".join(str(x) for x in a)
    print(s); LOG.append(s)

SEEDS = [42, 123, 2026]
ARCHS = ["resnet50", "mobilenetv3_large"]
K = 38

# ---------------- FROZEN BEFORE ANY TEST RESULT EXISTS ----------------
CONFIG = {
    "purpose": "Member 3: single test-set evaluation + near-duplicate leakage check",
    "image_size": 224,
    "preprocessing": "PIL open -> RGB -> resize 224x224 bilinear -> /255 -> ImageNet mean/std (identical to training code)",
    "imagenet_mean": [0.485, 0.456, 0.406],
    "imagenet_std": [0.229, 0.224, 0.225],
    "inference": "model.eval(), torch.autocast float16 on GPU (same as training validation), batch 256",
    "seeds": SEEDS,
    "models": {
        "resnet50": "runs_finetune_p10/resnet50/frac100_seed{seed}/best_model.pt",
        "mobilenetv3_large": "results/mobilenetv3/mobilenetv3_large_uniform_p010_seed{seed}/best_weights.pt",
    },
    "training_protocol_of_both_models": "full fine-tuning, uniform_p010, 3 epochs, AdamW backbone 1e-5 / head 5e-4, wd 1e-4, batch 32, fp16, no scheduler, cross-entropy; best epoch by validation macro-F1",
    "near_duplicate_rule": {
        "feature_extractor": "torchvision ResNet-50, ImageNet weights (DEFAULT), global-average-pooled 2048-d, L2-normalised",
        "primary_flag": "a test image is flagged if cosine similarity to its most similar TRAINING image >= 0.95",
        "cosine_threshold": 0.95,
        "second_signal": "64-bit difference hash (dHash 9x8 grayscale); reported separately, flagged if Hamming distance <= 4 bits",
        "hash_hamming_threshold": 4,
        "sensitivity_thresholds_for_information_only": [0.90, 0.95, 0.98, 0.99],
        "compared_against": "all 37,751 images with split == train",
        "note": "threshold chosen before looking at any test result and not changed afterwards",
    },
    "bootstrap": {"resamples": 2000, "rng_seed": 12345, "interval": "95% percentile",
                  "design": "test images resampled with replacement; the SAME resampled images are used for both models and all seeds; seed-mean macro-F1 is computed per resample"},
    "sd_definition": "sample standard deviation across the 3 seeds (ddof=1), same as the team's earlier tables",
    "low_support_class_warning_below_n_test_images": 50,
    "validation_reproduction": {"max_abs_macro_f1_difference": 0.003, "min_prediction_agreement_with_saved_file": 0.995},
}
EXPECTED_VAL_MACRO_F1 = {   # best_validation_macro_f1 from the teammates' training logs
    "resnet50": {42: 0.9837292878998758, 123: 0.978546, 2026: 0.981946},
    "mobilenetv3_large": {42: 0.9666848092339313, 123: 0.977196, 2026: 0.977689},
}
EXPECTED_SPLIT_SHA256 = "902f6a241712a1a37854ceee4b3833768e723288fd3d00e23f17576269aaf95c"
log("torch", torch.__version__, "| torchvision", torchvision.__version__, "| GPU", torch.cuda.get_device_name(0))

torch 2.11.0+cu130 | torchvision 0.26.0+cu130 | GPU Tesla T4


In [2]:
# ---------- 1. find every input file automatically ----------
def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

def usable(p):
    return "__MACOSX" not in p and not os.path.basename(p).startswith("._")

ALL, COLOR_CANDIDATES = [], []
for root, dirs, files in os.walk("/kaggle/input"):
    for d in dirs:
        if d == "color":
            COLOR_CANDIDATES.append(os.path.join(root, d))
    dirs[:] = [d for d in dirs if d not in ("color", "grayscale", "segmented")]
    for f in files:
        if f.lower().endswith((".zip", ".pt", ".csv", ".json")):
            ALL.append(os.path.join(root, f))

UNZ = WORK + "/unzipped"; os.makedirs(UNZ, exist_ok=True)
for z in [p for p in ALL if p.lower().endswith(".zip") and usable(p)]:
    dest = os.path.join(UNZ, os.path.splitext(os.path.basename(z))[0] + "_" + hashlib.md5(z.encode()).hexdigest()[:6])
    if not os.path.isdir(dest):
        with zipfile.ZipFile(z) as zf:
            zf.extractall(dest)
for root, _, files in os.walk(UNZ):
    for f in files:
        if f.lower().endswith((".pt", ".csv", ".json")):
            ALL.append(os.path.join(root, f))
ALL = [p for p in ALL if usable(p)]
log("files found:", len(ALL), "| color-folder candidates:", COLOR_CANDIDATES)

def find_one(pred, what):
    hits = sorted(p for p in ALL if pred(p))
    if not hits:
        raise RuntimeError(f"MISSING: {what}. Add it to the Kaggle dataset and run again.")
    if len({sha256(p) for p in hits}) > 1:
        raise RuntimeError(f"{what}: several DIFFERENT files match, remove the wrong one:\n" + "\n".join(hits))
    return hits[0]

CKPT = {}
for s in SEEDS:
    CKPT[("resnet50", s)] = find_one(lambda p, s=s: p.endswith("best_model.pt") and "finetune_p10" in p and "/resnet50/" in p and f"frac100_seed{s}/" in p,
                                     f"ResNet-50 fine-tune checkpoint, seed {s}")
    CKPT[("mobilenetv3_large", s)] = find_one(lambda p, s=s: p.endswith("best_weights.pt") and f"mobilenetv3_large_uniform_p010_seed{s}/" in p,
                                              f"MobileNetV3 checkpoint, seed {s}")

split_hashes = {}
for p in ALL:
    if p.endswith(".csv"):
        try:
            cols = pd.read_csv(p, nrows=0).columns.tolist()
            if {"path", "class", "split"} <= set(cols) and (pd.read_csv(p, usecols=["split"])["split"] == "test").any():
                split_hashes.setdefault(sha256(p), p)
        except Exception:
            pass
if len(split_hashes) != 1:
    raise RuntimeError(f"Need exactly one split CSV that contains test rows, found {len(split_hashes)}: {list(split_hashes.values())}")
SPLIT_PATH = list(split_hashes.values())[0]
SPLIT_SHA = list(split_hashes)[0]
if SPLIT_SHA != EXPECTED_SPLIT_SHA256:
    log("WARNING: split CSV hash differs from the team's original:", SPLIT_SHA)

df = pd.read_csv(SPLIT_PATH)
assert df.split.value_counts().to_dict() == {"train": 37751, "val": 7976, "test": 7978}, df.split.value_counts().to_dict()

COLOR_DIR = None
for c in COLOR_CANDIDATES:
    if all(os.path.exists(os.path.join(c, p)) for p in df.path.sample(200, random_state=0)):
        COLOR_DIR = c; break
if COLOR_DIR is None:
    raise RuntimeError("Could not find the PlantVillage 'color' folder (Abdallah Ali dataset). Add it as an input.")
missing = [p for p in df.path if not os.path.exists(os.path.join(COLOR_DIR, p))]
assert not missing, f"{len(missing)} images from the split file are missing, e.g. {missing[:3]}"
log("color folder:", COLOR_DIR); log("split file:", SPLIT_PATH); log("all 53,705 images found")

# class order = the order saved inside the ResNet checkpoint (same as class_to_idx.json)
ck = torch.load(CKPT[("resnet50", 42)], map_location="cpu", weights_only=False)
CLASS_TO_IDX = ck["class_to_idx"]; del ck
assert len(CLASS_TO_IDX) == K and set(CLASS_TO_IDX) == set(df["class"].unique())
for p in ALL:
    if os.path.basename(p) == "class_to_idx.json":
        assert json.load(open(p)) == CLASS_TO_IDX, f"class order in {p} differs from the checkpoint!"
CLASSES = [None] * K
for c, i in CLASS_TO_IDX.items():
    CLASSES[i] = c
df["y"] = df["class"].map(CLASS_TO_IDX).astype(int)
train_df = df[df.split == "train"].reset_index(drop=True)
val_df = df[df.split == "val"].reset_index(drop=True)
test_df = df[df.split == "test"].reset_index(drop=True)
y_val, y_test = val_df.y.values.astype(np.int64), test_df.y.values.astype(np.int64)
log("train/val/test:", len(train_df), len(val_df), len(test_df))

CONFIG["provenance"] = {
    "split_csv_sha256": SPLIT_SHA,
    "checkpoint_sha256": {f"{a}_seed{s}": sha256(CKPT[(a, s)]) for a in ARCHS for s in SEEDS},
    "checkpoint_files": {f"{a}_seed{s}": CKPT[(a, s)] for a in ARCHS for s in SEEDS},
    "class_to_idx": CLASS_TO_IDX,
    "environment": {"python": platform.python_version(), "torch": torch.__version__, "torchvision": torchvision.__version__,
                    "numpy": np.__version__, "gpu": torch.cuda.get_device_name(0)},
}
json.dump(CONFIG, open(OUT + "/config_test_evaluation.json", "w"), indent=2)
test_df[["path", "class", "y"]].rename(columns={"class": "true_class", "y": "true_index"}).to_csv(OUT + "/test_image_order.csv", index=False)
json.dump(CLASS_TO_IDX, open(OUT + "/class_to_idx.json", "w"), indent=2)

files found: 79 | color-folder candidates: ['/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color', '/kaggle/input/datasets/abdallahalidev/plantvillage-dataset/plantvillage dataset/color']
color folder: /kaggle/input/datasets/abdallahalidev/plantvillage-dataset/color
split file: /kaggle/input/datasets/shahnawaz006/test-evaluation-input/split (1).csv
all 53,705 images found
train/val/test: 37751 7976 7978


In [3]:
# ---------- 2. load images once (validation + test kept in memory), features for the near-duplicate check ----------
MEAN = torch.tensor(CONFIG["imagenet_mean"], device=DEVICE).view(1, 3, 1, 1)
STD = torch.tensor(CONFIG["imagenet_std"], device=DEVICE).view(1, 3, 1, 1)
AUTOCAST = lambda: torch.autocast("cuda", dtype=torch.float16)

class ImgDS(Dataset):
    def __init__(self, rels): self.rels = list(rels)
    def __len__(self): return len(self.rels)
    def __getitem__(self, i):
        with Image.open(os.path.join(COLOR_DIR, self.rels[i])) as im:
            im = im.convert("RGB").resize((224, 224), Image.Resampling.BILINEAR)
            arr = np.asarray(im, dtype=np.uint8).copy()
            g = np.asarray(im.convert("L").resize((9, 8), Image.Resampling.LANCZOS), dtype=np.int16)
        bits = (g[:, 1:] > g[:, :-1]).astype(np.uint8).reshape(-1)      # 64-bit dHash
        return torch.from_numpy(arr), torch.from_numpy(bits)

def to_input(x_u8):
    x = x_u8.to(DEVICE, non_blocking=True).permute(0, 3, 1, 2).float().div_(255.0)
    return (x - MEAN) / STD

FE = torchvision.models.resnet50(weights=torchvision.models.ResNet50_Weights.DEFAULT)
FE.fc = nn.Identity(); FE.eval().to(DEVICE)

def load_pass(rels, keep_images, want_features, name):
    t0 = time.time(); n = len(rels)
    imgs = np.empty((n, 224, 224, 3), np.uint8) if keep_images else None
    feats = np.empty((n, 2048), np.float16) if want_features else None
    bits = np.empty((n, 64), np.uint8)
    dl = DataLoader(ImgDS(rels), batch_size=128, shuffle=False, num_workers=min(4, os.cpu_count() or 1), pin_memory=True)
    i = 0
    with torch.no_grad():
        for xb, bb in dl:
            j = i + len(xb)
            if keep_images: imgs[i:j] = xb.numpy()
            bits[i:j] = bb.numpy()
            if want_features:
                with AUTOCAST():
                    f = FE(to_input(xb))
                feats[i:j] = F.normalize(f.float(), dim=1).cpu().numpy().astype(np.float16)
            i = j
    log(f"{name}: {n} images in {time.time()-t0:.0f}s")
    return imgs, feats, bits

val_imgs, _, _ = load_pass(val_df.path, True, False, "validation")
test_imgs, test_feats, test_bits = load_pass(test_df.path, True, True, "test (images kept, features for near-duplicate check)")
_, train_feats, train_bits = load_pass(train_df.path, False, True, "train (features only)")

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 216MB/s]


validation: 7976 images in 57s
test (images kept, features for near-duplicate check): 7978 images in 58s
train (features only): 37751 images in 259s


In [4]:
# ---------- 3. model loading + prediction ----------
def build(arch):
    if arch == "resnet50":
        m = torchvision.models.resnet50(weights=None); m.fc = nn.Linear(2048, K)
    else:
        m = torchvision.models.mobilenet_v3_large(weights=None); m.classifier[3] = nn.Linear(m.classifier[3].in_features, K)
    return m

def load_model(arch, seed):
    obj = torch.load(CKPT[(arch, seed)], map_location="cpu", weights_only=False)
    sd = obj
    if isinstance(obj, dict):
        for key in ("model_state_dict", "state_dict"):
            if key in obj: sd = obj[key]; break
    sd = {k.replace("module.", "", 1): v for k, v in sd.items()}
    m = build(arch); m.load_state_dict(sd, strict=True)
    return m.eval().to(DEVICE)

@torch.no_grad()
def predict_logits(model, imgs, bs=256):
    out = []
    for i in range(0, len(imgs), bs):
        x = to_input(torch.from_numpy(imgs[i:i + bs]))
        with AUTOCAST():
            out.append(model(x).float().cpu().numpy())
    return np.concatenate(out)

def macro_f1(y, p):
    tp = np.bincount(y[y == p], minlength=K).astype(float)
    den = np.bincount(y, minlength=K) + np.bincount(p, minlength=K)
    return float(np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0).mean())

def per_class_f1(y, p):
    tp = np.bincount(y[y == p], minlength=K).astype(float)
    den = np.bincount(y, minlength=K) + np.bincount(p, minlength=K)
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)

assert abs(macro_f1(y_val, (y_val + (np.arange(len(y_val)) % 7 == 0)) % K) -
           f1_score(y_val, (y_val + (np.arange(len(y_val)) % 7 == 0)) % K, labels=list(range(K)), average="macro", zero_division=0)) < 1e-9

In [5]:
# ---------- 4. VALIDATION reproduction check (test set is NOT touched here) ----------
tol = CONFIG["validation_reproduction"]["max_abs_macro_f1_difference"]
min_agree = CONFIG["validation_reproduction"]["min_prediction_agreement_with_saved_file"]
rows = []
for arch in ARCHS:
    for s in SEEDS:
        m = load_model(arch, s)
        p = predict_logits(m, val_imgs).argmax(1)
        f1 = macro_f1(y_val, p); exp = EXPECTED_VAL_MACRO_F1[arch][s]
        saved = [q for q in ALL if q.endswith("val_predictions.csv") and
                 ((arch == "resnet50" and "finetune_p10" in q and "/resnet50/" in q and f"frac100_seed{s}/" in q) or
                  (arch == "mobilenetv3_large" and f"mobilenetv3_large_uniform_p010_seed{s}/" in q))]
        agree = np.nan
        if saved:
            sv = pd.read_csv(saved[0]); col = "pred_class" if "pred_class" in sv.columns else "pred"
            mine = pd.DataFrame({"path": val_df.path, "mine": [CLASSES[i] for i in p]}).merge(sv[["path", col]], on="path")
            agree = float((mine["mine"] == mine[col]).mean())
        rows.append({"model": arch, "seed": s, "val_macro_f1_this_notebook": f1, "val_macro_f1_in_training_log": exp,
                     "abs_difference": abs(f1 - exp), "agreement_with_saved_val_predictions": agree})
        del m; torch.cuda.empty_cache()
vr = pd.DataFrame(rows); vr.to_csv(OUT + "/validation_reproduction_check.csv", index=False)
print(vr.to_string(index=False))
bad = vr[(vr.abs_difference > tol) | (vr.agreement_with_saved_val_predictions < min_agree)]
if len(bad):
    raise RuntimeError("STOP: the loaded models do not reproduce the validation numbers of the training logs. "
                       "The test set has NOT been used. Do not continue; send this table to Member 1/2.\n" + bad.to_string())
log("Validation check passed for all 6 models. Safe to continue to the test set.")

            model  seed  val_macro_f1_this_notebook  val_macro_f1_in_training_log  abs_difference  agreement_with_saved_val_predictions
         resnet50    42                    0.983597                      0.983729    1.323553e-04                              0.999875
         resnet50   123                    0.978407                      0.978546    1.385889e-04                              0.999875
         resnet50  2026                    0.981946                      0.981946    1.164697e-07                              1.000000
mobilenetv3_large    42                    0.966831                      0.966685    1.459178e-04                              0.999875
mobilenetv3_large   123                    0.977196                      0.977196    4.539967e-07                              1.000000
mobilenetv3_large  2026                    0.978150                      0.977689    4.609420e-04                              0.999749
Validation check passed for all 6 models. Safe t

In [6]:
# ---------- 5. near-duplicate check (rule frozen in CONFIG above) ----------
T_COS = CONFIG["near_duplicate_rule"]["cosine_threshold"]
T_HAM = CONFIG["near_duplicate_rule"]["hash_hamming_threshold"]
Ftr = torch.from_numpy(train_feats).to(DEVICE).float()
Btr = torch.from_numpy(train_bits).to(DEVICE).float(); Btr_sum = Btr.sum(1)
Nt = len(test_df)
max_cos = np.zeros(Nt, np.float32); arg_cos = np.zeros(Nt, np.int64)
min_ham = np.zeros(Nt, np.int64); arg_ham = np.zeros(Nt, np.int64)
for i in range(0, Nt, 512):
    q = torch.from_numpy(test_feats[i:i + 512]).to(DEVICE).float()
    v, a = (q @ Ftr.T).max(1)
    max_cos[i:i + 512] = v.cpu().numpy(); arg_cos[i:i + 512] = a.cpu().numpy()
    qb = torch.from_numpy(test_bits[i:i + 512]).to(DEVICE).float()
    ham = qb.sum(1)[:, None] + Btr_sum[None, :] - 2 * (qb @ Btr.T)
    v, a = ham.min(1)
    min_ham[i:i + 512] = v.cpu().numpy().round().astype(np.int64); arg_ham[i:i + 512] = a.cpu().numpy()
del Ftr, Btr; torch.cuda.empty_cache()

FLAG = max_cos >= T_COS                      # primary rule
FLAG_HASH = min_ham <= T_HAM                 # second signal
nd = pd.DataFrame({
    "path": test_df.path, "true_class": test_df["class"],
    "max_cosine_to_train": max_cos, "nearest_train_path": train_df.path.values[arg_cos],
    "nearest_train_class": train_df["class"].values[arg_cos],
    "nearest_has_same_class": train_df["class"].values[arg_cos] == test_df["class"].values,
    "min_hash_hamming_bits": min_ham, "nearest_hash_train_path": train_df.path.values[arg_ham],
    "flag_cosine_primary": FLAG, "flag_hash": FLAG_HASH, "flag_either": FLAG | FLAG_HASH})
nd.to_csv(OUT + "/near_duplicate/test_near_duplicate_flags.csv", index=False)
nd_summary = {
    "rule": "flagged if cosine similarity to nearest training image >= %.2f" % T_COS,
    "n_test_images": int(Nt), "n_flagged_cosine_primary": int(FLAG.sum()), "share_flagged_cosine_primary": float(FLAG.mean()),
    "n_flagged_hash": int(FLAG_HASH.sum()), "n_flagged_both_signals": int((FLAG & FLAG_HASH).sum()), "n_flagged_either": int((FLAG | FLAG_HASH).sum()),
    "flagged_by_threshold_information_only": {str(t): int((max_cos >= t).sum()) for t in CONFIG["near_duplicate_rule"]["sensitivity_thresholds_for_information_only"]},
    "median_max_cosine": float(np.median(max_cos)),
    "nearest_train_image_has_different_class_among_flagged": int((FLAG & ~nd.nearest_has_same_class.values).sum()),
}
json.dump(nd_summary, open(OUT + "/near_duplicate/near_duplicate_summary.json", "w"), indent=2)
log("near-duplicates (primary rule):", nd_summary["n_flagged_cosine_primary"], "of", Nt, "| by hash:", nd_summary["n_flagged_hash"], "| both:", nd_summary["n_flagged_both_signals"])

near-duplicates (primary rule): 353 of 7978 | by hash: 53 | both: 13


In [7]:
# ---------- 6. THE TEST RUN (once per model per seed) ----------
LOCK = WORK + "/test_run_lock.json"
if os.path.exists(LOCK):
    raise RuntimeError("The test set was already evaluated in this session. Not running it again. Restart the notebook only if you really must.")

def softmax(z):
    z = z - z.max(1, keepdims=True); e = np.exp(z); return e / e.sum(1, keepdims=True)

TEST_LOGITS = {}
for arch in ARCHS:
    for s in SEEDS:
        m = load_model(arch, s)
        lg = predict_logits(m, test_imgs)
        TEST_LOGITS[(arch, s)] = lg
        p = lg.argmax(1)
        np.save(OUT + f"/predictions/test_logits_{arch}_seed{s}.npy", lg.astype(np.float32))
        pd.DataFrame({"path": test_df.path, "y_true": y_test, "y_pred": p,
                      "true_class": test_df["class"], "pred_class": [CLASSES[i] for i in p],
                      "confidence": softmax(lg).max(1), "correct": p == y_test,
                      "flag_near_duplicate": FLAG, "max_cosine_to_train": max_cos}
                     ).to_csv(OUT + f"/predictions/test_predictions_{arch}_seed{s}.csv", index=False)
        log(f"test | {arch} | seed {s} | macro-F1 {macro_f1(y_test, p):.4f} | accuracy {(p == y_test).mean():.4f}")
        del m; torch.cuda.empty_cache()
json.dump({"done": time.strftime("%Y-%m-%d %H:%M:%S")}, open(LOCK, "w"))

test | resnet50 | seed 42 | macro-F1 0.9834 | accuracy 0.9875
test | resnet50 | seed 123 | macro-F1 0.9795 | accuracy 0.9865
test | resnet50 | seed 2026 | macro-F1 0.9834 | accuracy 0.9878
test | mobilenetv3_large | seed 42 | macro-F1 0.9537 | accuracy 0.9673
test | mobilenetv3_large | seed 123 | macro-F1 0.9685 | accuracy 0.9787
test | mobilenetv3_large | seed 2026 | macro-F1 0.9709 | accuracy 0.9796


In [8]:
# ---------- 7. statistics ----------
N = len(y_test); KEEP = ~FLAG
B = CONFIG["bootstrap"]["resamples"]
rng = np.random.default_rng(CONFIG["bootstrap"]["rng_seed"])

def boot_macro_f1(y, p, idx):
    yb, pb = y[idx], p[idx]
    off = np.arange(idx.shape[0])[:, None] * K
    tp = np.bincount((yb + off)[yb == pb], minlength=idx.shape[0] * K).reshape(-1, K)
    tc = np.bincount((yb + off).ravel(), minlength=idx.shape[0] * K).reshape(-1, K)
    pc = np.bincount((pb + off).ravel(), minlength=idx.shape[0] * K).reshape(-1, K)
    den = tc + pc
    return np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0).mean(1)

def ci(x): return float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5))

PRED = {k: v.argmax(1) for k, v in TEST_LOGITS.items()}
IDX_ALL = rng.integers(0, N, size=(B, N), dtype=np.int32)                 # same images for every model and seed
n_keep = int(KEEP.sum())
IDX_KEEP = rng.integers(0, n_keep, size=(B, n_keep), dtype=np.int32)

# per run table
rows = []
for (arch, s), p in PRED.items():
    rows.append({"model": arch, "seed": s, "n_test": N,
                 "test_macro_f1": macro_f1(y_test, p), "test_accuracy": float((p == y_test).mean()),
                 "n_flagged_near_duplicates": int(FLAG.sum()),
                 "macro_f1_excluding_flagged": macro_f1(y_test[KEEP], p[KEEP]),
                 "accuracy_flagged_images": float((p[FLAG] == y_test[FLAG]).mean()) if FLAG.any() else np.nan,
                 "accuracy_unflagged_images": float((p[KEEP] == y_test[KEEP]).mean())})
per_seed = pd.DataFrame(rows); per_seed.to_csv(OUT + "/test_results_per_seed.csv", index=False)

# bootstrap
BM_ALL = {k: boot_macro_f1(y_test, p, IDX_ALL) for k, p in PRED.items()}
BM_KEEP = {k: boot_macro_f1(y_test[KEEP], p[KEEP], IDX_KEEP) for k, p in PRED.items()}
def seedmean(BM, arch): return np.mean([BM[(arch, s)] for s in SEEDS], axis=0)

summ, cirows = [], []
for arch in ARCHS:
    d = per_seed[per_seed.model == arch]
    lo, hi = ci(seedmean(BM_ALL, arch)); lo2, hi2 = ci(seedmean(BM_KEEP, arch))
    summ.append({"model": arch, "test_macro_f1_mean": d.test_macro_f1.mean(), "test_macro_f1_sd": d.test_macro_f1.std(ddof=1),
                 "ci95_low_seed_mean": lo, "ci95_high_seed_mean": hi,
                 "test_accuracy_mean": d.test_accuracy.mean(),
                 "n_flagged_near_duplicates": int(FLAG.sum()),
                 "macro_f1_excluding_flagged_mean": d.macro_f1_excluding_flagged.mean(),
                 "macro_f1_excluding_flagged_sd": d.macro_f1_excluding_flagged.std(ddof=1),
                 "ci95_low_excluding_flagged": lo2, "ci95_high_excluding_flagged": hi2})
    cirows.append({"model": arch, "scope": "all_test_images", "quantity": "seed-mean macro-F1", "point": d.test_macro_f1.mean(), "ci95_low": lo, "ci95_high": hi})
    cirows.append({"model": arch, "scope": "excluding_flagged", "quantity": "seed-mean macro-F1", "point": d.macro_f1_excluding_flagged.mean(), "ci95_low": lo2, "ci95_high": hi2})
    for s in SEEDS:
        l, h = ci(BM_ALL[(arch, s)])
        cirows.append({"model": arch, "scope": "all_test_images", "quantity": f"macro-F1 seed {s}", "point": macro_f1(y_test, PRED[(arch, s)]), "ci95_low": l, "ci95_high": h})
summary = pd.DataFrame(summ); summary.to_csv(OUT + "/test_results_summary.csv", index=False)
pd.DataFrame(cirows).to_csv(OUT + "/bootstrap_confidence_intervals.csv", index=False)

# paired comparison on the same resampled images
A, M = ARCHS
prow = []
for scope, BM in (("all_test_images", BM_ALL), ("excluding_flagged", BM_KEEP)):
    D = seedmean(BM, A) - seedmean(BM, M); l, h = ci(D)
    pt = (per_seed[per_seed.model == A].test_macro_f1.mean() - per_seed[per_seed.model == M].test_macro_f1.mean()) if scope == "all_test_images" else \
         (per_seed[per_seed.model == A].macro_f1_excluding_flagged.mean() - per_seed[per_seed.model == M].macro_f1_excluding_flagged.mean())
    prow.append({"scope": scope, "quantity": f"{A} minus {M}, seed-mean macro-F1", "point_difference": pt, "ci95_low": l, "ci95_high": h,
                 "ci_excludes_zero": bool(l > 0 or h < 0), "share_of_resamples_where_resnet50_is_better": float((D > 0).mean())})
for s in SEEDS:
    D = BM_ALL[(A, s)] - BM_ALL[(M, s)]; l, h = ci(D)
    pa, pm = PRED[(A, s)], PRED[(M, s)]; ca, cm = pa == y_test, pm == y_test
    prow.append({"scope": "all_test_images", "quantity": f"{A} minus {M}, macro-F1 seed {s}",
                 "point_difference": macro_f1(y_test, pa) - macro_f1(y_test, pm), "ci95_low": l, "ci95_high": h,
                 "ci_excludes_zero": bool(l > 0 or h < 0), "share_of_resamples_where_resnet50_is_better": float((D > 0).mean()),
                 "both_correct": int((ca & cm).sum()), "only_resnet50_correct": int((ca & ~cm).sum()),
                 "only_mobilenetv3_correct": int((~ca & cm).sum()), "both_wrong": int((~ca & ~cm).sum())})
paired = pd.DataFrame(prow); paired.to_csv(OUT + "/paired_comparison.csv", index=False)

# near-duplicate sensitivity (information only, rule above stays the primary one)
srow = []
for t in CONFIG["near_duplicate_rule"]["sensitivity_thresholds_for_information_only"]:
    k = max_cos < t
    for arch in ARCHS:
        srow.append({"cosine_threshold": t, "model": arch, "n_flagged": int((~k).sum()),
                     "macro_f1_excluding_flagged_mean_over_seeds": float(np.mean([macro_f1(y_test[k], PRED[(arch, s)][k]) for s in SEEDS]))})
pd.DataFrame(srow).to_csv(OUT + "/near_duplicate/sensitivity_to_threshold_information_only.csv", index=False)

# per class
support = np.bincount(y_test, minlength=K)
pc = pd.DataFrame({"class": CLASSES, "n_test_images": support})
for arch in ARCHS:
    pc[f"f1_{arch}_mean_over_seeds"] = np.mean([per_class_f1(y_test, PRED[(arch, s)]) for s in SEEDS], axis=0)
pc["low_support_warning"] = pc.n_test_images < CONFIG["low_support_class_warning_below_n_test_images"]
pc.to_csv(OUT + "/per_class_test_f1.csv", index=False)

pd.set_option("display.width", 200)
print(per_seed.round(4).to_string(index=False)); print(); print(summary.round(4).T.to_string()); print(); print(paired.round(4).iloc[:2].T.to_string())
print("\nclasses with fewer than 50 test images:", pc[pc.low_support_warning]["class"].tolist())

            model  seed  n_test  test_macro_f1  test_accuracy  n_flagged_near_duplicates  macro_f1_excluding_flagged  accuracy_flagged_images  accuracy_unflagged_images
         resnet50    42    7978         0.9834         0.9875                        353                      0.9830                   1.0000                     0.9869
         resnet50   123    7978         0.9795         0.9865                        353                      0.9793                   1.0000                     0.9858
         resnet50  2026    7978         0.9834         0.9878                        353                      0.9832                   1.0000                     0.9873
mobilenetv3_large    42    7978         0.9537         0.9673                        353                      0.9533                   1.0000                     0.9658
mobilenetv3_large   123    7978         0.9685         0.9787                        353                      0.9682                   0.9972              

In [9]:
# ---------- 8. README for the next person + zip ----------
s_all = summary.set_index("model")
def fmt(a, b, c, d): return f"{a:.4f} ± {b:.4f} (95% CI {c:.4f}–{d:.4f})"
pr = paired.iloc[0]
readme = f"""# Member 3 handover: PlantVillage test evaluation and near-duplicate check

Everything here was produced by ONE run of the notebook. The test set was used exactly once per model per seed
(2 models x 3 seeds = 6 evaluations). Nothing was changed after seeing test results.

## Main numbers (test set, {N} images, 38 classes)
| Model | Test macro-F1, mean ± SD over 3 seeds (95% bootstrap CI of the seed mean) | After removing near-duplicates |
|---|---|---|
| ResNet-50 | {fmt(s_all.loc['resnet50','test_macro_f1_mean'], s_all.loc['resnet50','test_macro_f1_sd'], s_all.loc['resnet50','ci95_low_seed_mean'], s_all.loc['resnet50','ci95_high_seed_mean'])} | {s_all.loc['resnet50','macro_f1_excluding_flagged_mean']:.4f} ± {s_all.loc['resnet50','macro_f1_excluding_flagged_sd']:.4f} |
| MobileNetV3-Large | {fmt(s_all.loc['mobilenetv3_large','test_macro_f1_mean'], s_all.loc['mobilenetv3_large','test_macro_f1_sd'], s_all.loc['mobilenetv3_large','ci95_low_seed_mean'], s_all.loc['mobilenetv3_large','ci95_high_seed_mean'])} | {s_all.loc['mobilenetv3_large','macro_f1_excluding_flagged_mean']:.4f} ± {s_all.loc['mobilenetv3_large','macro_f1_excluding_flagged_sd']:.4f} |

Paired comparison on the same resampled test images (ResNet-50 minus MobileNetV3): {pr.point_difference:+.4f}, 95% CI {pr.ci95_low:+.4f} to {pr.ci95_high:+.4f}
({'the interval excludes zero' if pr.ci_excludes_zero else 'the interval includes zero, so no clear winner'}).

Near-duplicates: {nd_summary['n_flagged_cosine_primary']} of {N} test images ({100*nd_summary['share_flagged_cosine_primary']:.1f}%) have a training image with cosine similarity >= {T_COS}
(ImageNet ResNet-50 features). The threshold was fixed before any test result. The perceptual-hash signal flagged {nd_summary['n_flagged_hash']} images, {nd_summary['n_flagged_both_signals']} by both.

## Files
- `test_results_summary.csv`, `test_results_per_seed.csv`: the numbers above, per seed and averaged.
- `bootstrap_confidence_intervals.csv`, `paired_comparison.csv`: confidence intervals and the model-vs-model comparison.
- `per_class_test_f1.csv`: per-class F1 and number of test images (classes under 50 images are marked; their F1 is noisy).
- `predictions/test_predictions_<model>_seed<seed>.csv`: path, true/predicted class, confidence, near-duplicate flag for every test image.
- `predictions/test_logits_<model>_seed<seed>.npy`: raw model outputs, shape (7978, 38), same row order as `test_image_order.csv`.
- `near_duplicate/`: per-image flags with the nearest training image, summary, and a threshold-sensitivity table (information only).
- `validation_reproduction_check.csv`: proof that the notebook reproduced the teammates' validation scores before touching test.
- `config_test_evaluation.json`: every setting, the near-duplicate rule, hashes of all checkpoints and the split file, software versions.
- `class_to_idx.json`, `test_image_order.csv`, `run_log.txt`.

## For Member 4 (PlantDoc field test)
Do not run the models on the lab images again. Read `predictions/test_predictions_<model>_seed<seed>.csv`, keep the classes that PlantDoc covers,
and compute lab macro-F1 on those classes only. Use `flag_near_duplicate == False` if you want the leakage-free lab score.

## Honest limits
- Only 3 seeds; the SD is a sample SD (n-1). The bootstrap interval reflects which test images were drawn, not extra seeds.
- Near-duplicate detection is a similarity rule, not proof. Different thresholds give different counts (see the sensitivity table).
- Several classes have very few test images (see `per_class_test_f1.csv`).
- Both models were fully fine-tuned with `uniform_p010`, 3 epochs; results do not apply to other training setups.
"""
open(OUT + "/README_handover.md", "w").write(readme)
open(OUT + "/run_log.txt", "w").write("\n".join(LOG))
zip_path = shutil.make_archive(WORK + "/terravista_member3_test_evaluation", "zip", WORK, "terravista_member3_test_evaluation")
log("ZIP READY:", zip_path, f"({os.path.getsize(zip_path)/1e6:.1f} MB)")
print(readme[:1500])

ZIP READY: /kaggle/working/terravista_member3_test_evaluation.zip (7.0 MB)
# Member 3 handover: PlantVillage test evaluation and near-duplicate check

Everything here was produced by ONE run of the notebook. The test set was used exactly once per model per seed
(2 models x 3 seeds = 6 evaluations). Nothing was changed after seeing test results.

## Main numbers (test set, 7978 images, 38 classes)
| Model | Test macro-F1, mean ± SD over 3 seeds (95% bootstrap CI of the seed mean) | After removing near-duplicates |
|---|---|---|
| ResNet-50 | 0.9821 ± 0.0022 (95% CI 0.9787–0.9848) | 0.9818 ± 0.0022 |
| MobileNetV3-Large | 0.9644 ± 0.0093 (95% CI 0.9592–0.9686) | 0.9640 ± 0.0094 |

Paired comparison on the same resampled test images (ResNet-50 minus MobileNetV3): +0.0178, 95% CI +0.0136 to +0.0220
(the interval excludes zero).

Near-duplicates: 353 of 7978 test images (4.4%) have a training image with cosine similarity >= 0.95
(ImageNet ResNet-50 features). The threshold was fixed before 